# 다시 쓰는 전시

**오늘의 역할** · 전시를 운영하는 학생입니다. 같은 운영 메모로 관람객용 카드를 두 장 만들고, 검증과 원문을 보고 한 장을 고르거나 게시를 보류합니다.

**도입** · BAG-07의 두 출력은 사실값을 모두 맞힙니다. rank 2는 운영 메모를 다시 적고 rank 4는 한 문장을 냅니다. TEE-11에서는 rank 4의 자유 문장이 반복됩니다. 예시 앱에서 모델 원문을 열고 내 기준을 한 줄로 적습니다.

**실습 순서** · 응답 마스크 → rank 학습 → temperature 생성 → 게시 판정 → 내 운영 메모. CPU를 선택합니다. 의존성이 없는 환경은 아래 설치 셀이 먼저 준비합니다. 모델을 불러오는 동안 01의 작은 계산부터 풉니다.

**목표 카드 예시** · 헌 현수막과 면 끈이 다시 쓸 수 있는 토트백이 됐어요. 디자인실에서 두 재료가 어떻게 한 가방에 쓰였는지 살펴보세요. 이 문장은 수업에 제시하는 목표 예입니다.


In [ ]:
PACKAGE_BYTES = '...'
import base64
import io
import os
import sys
import tempfile
import zipfile
from pathlib import Path
from IPython.display import display

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

if colab_files is not None:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.15.1", "peft==0.21.2", "ipywidgets>=8,<9"], check=True)

workspace = Path(tempfile.mkdtemp(prefix="w13b-cards-"))
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE_BYTES))) as archive:
    archive.extractall(workspace)
sys.path.insert(0, str(workspace))
from luna_genai.exhibit_studio import ExhibitTrainer, FIELDS, ROOT, dataset, load_json
from luna_genai.exhibit_showcase import TEMPLATE
from luna_genai.exhibit_showcase import (AUDIENCES, MEMO_SCHEMA, fit_pair, lesson_cases,
    measures, memo_case, rank_items, replay_items, temperature_item, valid_sampling,
    validate_design, write_artifacts)

train = [case for case in dataset() if case["split"] == "train"]
train_by_id = {case["id"]: case for case in train}
common_train = [train_by_id[key] for key in ("BAG-01", "SHO-01", "TEE-01", "BAG-04")]
validate_cases = lesson_cases()
reference = replay_items()
print(f"학습 {len(train)}건의 고정 순서 · 검증 " + ", ".join(c["id"] for c in validate_cases))


## 01 · 무엇을 학습할지 정하기

Decoder-only Transformer에서 현 위치의 로짓은 그 다음 위치의 토큰을 맞힙니다. 입력 11, 12 뒤에 정답 21, 22를 붙이면, 12 다음의 로짓이 21을 평가합니다. 모델은 11과 12를 문맥으로 읽고, 그 자리를 정답으로 세지 않습니다. 라벨에서 -100인 위치를 손실 계산에서 제외합니다.

$$L_\mathrm{response}=-\frac{1}{|R|}\sum_{t\in R}\log p_\theta(y_t\mid x,y_{<t}).$$

**예측** · 두 후보가 첫 응답 토큰에 각각 0.5와 0.25를 줬습니다. 어느 손실이 더 큰지 계산한 뒤 다음 셀 출력과 비교하세요.

**입력 → 호출 → 반환** · prefix [11, 12]와 answer [21, 22] → make_labels → [-100, -100, 21, 22].

**힌트** · ① 앞자리 개수는 prefix 길이입니다. ② -100을 앞자리 길이만큼 반복하세요. ③ 정답 토큰을 순서대로 붙이세요. 빈 prefix와 길이가 다른 예에도 같은 코드가 동작해야 합니다.

[W03B 다음 토큰](https://github.com/lunalab-ai/genAI/blob/2026-fall-w03b/course/notion/w03b/w03b-transformer-walkthrough.md#L156) · [W09B 지시 학습 손실](https://github.com/lunalab-ai/genAI/blob/2026-fall-w09b/course/notion/w09b/w09b-instructions.md#L33) · [W10A 마스크 코드](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L203)


In [ ]:
def make_labels(prefix, answer):
    # TODO 1. 실제 학습의 labels를 만들어 반환하세요.
    return list(prefix) + list(answer)


In [ ]:
import math

print("첫 응답 한 토큰 손실", {p: round(-math.log(p), 3) for p in [0.5, 0.25]})
label_ready = (make_labels([11, 12], [21, 22]) == [-100, -100, 21, 22]
               and make_labels([7], [8, 9, 10]) == [-100, 8, 9, 10])
print("내 라벨", make_labels([11, 12], [21, 22]))
print("통과" if label_ready else "앞쪽 두 위치가 아직 손실 제외가 아닙니다. TODO 1을 확인하세요.")


## 02 · LoRA rank를 바꿔 직접 학습하기

$$W'=W+\frac{\alpha}{r}BA,\qquad \#(A,B)=r(d_\mathrm{in}+d_\mathrm{out}).$$

앞·뒤 차원이 모두 16이면 rank 2는 64개, rank 4는 128개를 학습합니다. W13B는 같은 Qwen 모델의 q_proj와 v_proj에 어댑터를 붙입니다. A/B는 **동일한 4개 훈련 사례, 순서, 학습률 0.0005, 시드 42, 배치 크기 1**을 사용합니다. 대상 4건을 보기 전에 학습 사례 ID와 검증 사례 ID를 비교하세요.

**입력 → 호출 → 반환** · A/B 설정 → fit_pair(..., make_labels, ...) → 학습 모수 수·어댑터 변화·사례별 손실.

**힌트** · ① A는 rank 2입니다. B는 rank가 두 배입니다. ② 두 모델의 업데이트 수는 4로 맞춥니다. ③ alpha를 rank에 맞춰 계산하여 alpha/r을 둘 다 2로 만듭니다.

**CPU 준비가 늦을 때** · RUN_CPU를 False로 놓고 03과 04로 넘어갈 수 있습니다. 04의 사전 기록은 24회 대 24회의 이전 CPU 실측입니다. 학습과 검증을 먼저 시작할 경우 모델 준비 시간은 기기에 따라 다릅니다.

[W10A rank와 alpha](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L99)


In [ ]:
def two_designs():
    rank_b = None  # TODO 2a · A와 비교할 B의 rank를 정하세요.
    steps_b = None  # TODO 2b · A와 같은 업데이트 수를 넣으세요.
    alpha_b = None  # TODO 2c · B의 LoRA 스케일을 A와 맞추세요.
    return [{"rank": 2, "steps": 4, "alpha": 4},
            {"rank": rank_b, "steps": steps_b, "alpha": alpha_b}]


In [ ]:
design = two_designs()
print("내 설계", design)
print("공통 학습 순서", [c["id"] for c in common_train])
print("검증 사례", [c["id"] for c in validate_cases])
RUN_CPU = True  # @param {type:"boolean"}
trainer = None
training = {}
if RUN_CPU and label_ready and validate_design(design):
    trainer = ExhibitTrainer()
    example = trainer.tokens(common_train[0])
    sample_labels = make_labels(example["prefix"], example["response"])
    print("프롬프트 ID 끝", example["prefix"][-8:])
    print("응답 ID 처음", example["response"][:8])
    print("학습 입력과 응답 경계 라벨", sample_labels[len(example["prefix"])-3:len(example["prefix"])+4])
    print("기본 모델의 첫 응답 토큰", trainer.first_token(example))
    training = fit_pair(trainer, common_train, design, make_labels, workspace / "trained")
else:
    print("내 학습 대기 · TODO 1·2를 완료한 뒤 이 셀을 다시 실행하세요. 사전 실측 비교는 04에서 열 수 있습니다.")


## 03 · 학습한 모델의 다음 토큰 선택을 조정하기

Temperature는 로짓을 T로 나눈 뒤 확률을 만듭니다. 같은 [2, 1, 0]에서 T가 0.35일 때 최댓값의 확률이 T가 0.85일 때보다 얼마나 큰지 보세요. 한 모델, 한 작품, 한 프롬프트 안에서 비교합니다. 두 조건 모두 샘플링을 켜고 시드를 42로 초기화합니다. top-k를 0으로 두고 top-p를 1.0으로 두어 이 두 제한을 해제합니다.

$$p_T(i)=\frac{\exp(z_i/T)}{\sum_j\exp(z_j/T)}.$$

**입력 → 호출 → 반환** · 온도 0.35 또는 0.85 → sampling → model.generate에 들어가는 설정.

**힌트** · ① greedy를 고르면 temperature를 비교할 수 없습니다. ② T 자리에 함수 인자를 넣으세요. ③ 다른 세대 조건은 같아야 합니다. 최대 새 토큰 수는 112입니다.

[Transformers 생성 설정](https://huggingface.co/docs/transformers/en/main_classes/text_generation)


In [ ]:
def sampling(temp):
    return {"do_sample": False, "temperature": 1.0,  # TODO 3. 생성 방식을 연결하세요.
            "top_k": 0, "top_p": 1.0, "max_new_tokens": 112}


In [ ]:
for temp in (0.35, 0.85):
    mass = [math.exp((x - 2) / temp) for x in (2, 1, 0)]
    print("toy T", temp, [round(x / sum(mass), 3) for x in mass])
sampling_ready = all(valid_sampling(sampling(t), t) for t in (0.35, 0.85))
print("샘플링 조건 통과" if sampling_ready else "TODO 3의 샘플링과 temperature를 확인하세요.")


## 04 · 사실을 맞힌 출력에도 게시 보류 적용하기

정답 토큰 NLL은 정답이 주어졌을 때의 평균 손실입니다. 생성 원문은 **사실 항목 4개**, **정확히 두 줄**, **같은 단어의 연속 반복**을 따로 봅니다. 예를 들어 TEE-11은 예약이 비어 있으므로 '확인 필요'가 정답입니다. 그 네 항목을 모두 맞혀도 소개 문장에 반복이 있으면 보류하세요.

**입력 → 호출 → 반환** · 일치 수, 형식 통과 여부, 반복 최대 길이 → can_publish → True 또는 False.

**힌트** · ① 네 항목이 모두 일치해야 합니다. ② 형식 검사도 통과해야 합니다. ③ 같은 단어가 세 번 연속이면 보류합니다. 네 입력 예의 판정을 종이에 먼저 적으세요.

공동 사례에 적용한 후 내 사례에도 같은 함수를 사용합니다. 게시 후보의 자유 문장은 원문을 한 번 더 읽습니다.

[W10A NLL·PPL과 출력 평가](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L220)


In [ ]:
def can_publish(matches, two_lines, longest_repeat):
    # TODO 4. 세 검사 결과를 읽어 게시 가능 여부를 반환하세요.
    return matches == 4


In [ ]:
def comparison_table(items):
    import pandas as pd
    records = []
    for item in items:
        for row in item["rows"]:
            records.append({"작품": item["key"], "실험": item["title"], "조건": row["name"],
                "업데이트": row["steps"], "사실": f"{row['matches']}/4",
                "형식": "통과" if row["strict"] else "실패", "최대 반복": row["repetition"],
                "정답 NLL": row["nll"], "게시": "검토 가능" if row["approved"] else "보류"})
    display(pd.DataFrame(records))

cases_for_check = [(4, True, 1), (4, False, 1), (4, True, 32), (3, True, 1)]
rule_ready = [can_publish(*args) for args in cases_for_check] == [True, False, False, False]
print("내 판정", [can_publish(*args) for args in cases_for_check])
items = list(reference)
if training and rule_ready:
    items = rank_items(trainer, validate_cases, can_publish) + items
    if sampling_ready:
        items.insert(1, temperature_item(trainer, validate_cases[0], 4, 4, sampling, can_publish))
else:
    print("수업 실측 재생 · 내 비교는 TODO가 준비되면 이 자리에 표시됩니다.")
comparison_table(items)
downloads = write_artifacts(workspace / "deliverables", items, "BAG-07")
print("BAG-07과 TEE-11을 선택해 두 카드의 운영 메모, 원문, 반복 수를 비교하세요.")


## 05 · 내 운영 메모로 카드 만들기

**작품명과 운영 메모를 직접 바꾸세요.** 예약 정보를 모르면 빈칸으로 남겨도 됩니다. 관람객을 골라 변경한 메모로 두 카드를 만듭니다. 한 프롬프트에서 temperature 두 값만 바꿉니다. 결과가 좋지 않으면 비교표의 빨간 항목을 확인하고 보류하세요.
내 4회 어댑터의 성능을 먼저 확인한 후 24회 수업 학습 어댑터를 선택해 같은 새 메모를 생성할 수 있습니다.

브라우저 HTML에서 만든 메모가 있으면 아래에서 JSON을 불러옵니다. 생성한 결과는 HTML로 열거나 JSON을 내려받아 브라우저 앱에 가져갈 수 있습니다.

**기록 한 줄** · 내가 고른 설정 / 결정적 증거 / 채택 또는 보류.

[과제 1의 24회·9건용 별도 Colab](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w13b/notebooks/student/w13b_exhibit.ipynb)


In [ ]:
import json
import ipywidgets as w
from IPython.display import FileLink, clear_output

def mount_studio():
    case = validate_cases[0]
    widths = {"layout": w.Layout(width="100%"), "style": {"description_width": "120px"}}
    title = w.Text(description="작품명", value=case["title"], **widths)
    material = w.Textarea(description="재료와 리폼 과정", value=case["material"], **widths)
    fields = {name: w.Text(description=name, value=case["facts"][name], **widths) for name in FIELDS}
    audience = w.Dropdown(description="소개할 관람객", options=AUDIENCES, **widths)
    rank_select = w.Dropdown(description="사용할 어댑터", options=[
        ("수업 학습 rank 4 · 24회", (4, 24)),
        ("내가 학습한 rank 2 · 4회", (2, 4)),
        ("내가 학습한 rank 4 · 4회", (4, 4)),
    ], **widths)
    upload = w.FileUpload(description="운영 메모 JSON", accept=".json", multiple=False)
    run = w.Button(description="이 메모로 두 카드 생성", button_style="primary", layout=w.Layout(width="220px"))
    html_button = w.Button(description="비교 HTML 받기", layout=w.Layout(width="160px"))
    result_button = w.Button(description="결과 JSON 받기", layout=w.Layout(width="160px"))
    zip_button = w.Button(description="앱 ZIP 받기", layout=w.Layout(width="160px"))
    output = w.Output()

    def load_memo(change):
        if not change["new"]:
            return
        with output:
            clear_output()
            try:
                memo = json.loads(bytes(change["new"][0]["content"]))
                saved = memo_case(memo)
                title.value, material.value, audience.value = saved["title"], saved["material"], saved["audience"]
                for f, value in saved["facts"].items():
                    fields[f].value = "" if value == "확인 필요" else value
                print("운영 메모를 불러왔습니다.")
            except (ValueError, TypeError, KeyError, AttributeError) as exc:
                print("메모 JSON을 확인하세요.", exc)

    def current_memo():
        return {"schema": MEMO_SCHEMA, "title": title.value, "material": material.value,
                "audience": audience.value, "facts": {f: control.value for f, control in fields.items()}}

    def generate(_):
        global downloads
        with output:
            clear_output()
            try:
                if not (trainer and training and rule_ready and sampling_ready):
                    print("새 메모를 생성하려면 TODO 1–4를 완료하고 02와 04를 실행하세요.")
                    return
                new_case = memo_case(current_memo())
                print("동일 메모에서 두 조건으로 생성 중", new_case["id"], flush=True)
                rank, steps = rank_select.value
                if steps == 24 and (rank, steps) not in trainer.adapter_paths:
                    recorded = load_json(ROOT / "reference.json")["training"][str(rank)]
                    trainer.register_adapter(rank, steps, TEMPLATE.parent / "trained-rank4", recorded)
                item = temperature_item(trainer, new_case, rank, steps, sampling,
                                        can_publish, audience_prompt=True)
                downloads = write_artifacts(workspace / "deliverables", [item, *items], new_case["id"])
                comparison_table([item])
                for row in item["rows"]:
                    print(row["name"], "\n", row["raw"])
                print("비교 HTML 또는 결과 JSON을 받으세요.")
            except (ValueError, RuntimeError) as exc:
                print(exc)

    def download(index):
        with output:
            path = downloads[index]
            if colab_files is not None:
                colab_files.download(str(path))
            else:
                display(FileLink(str(path)))

    upload.observe(load_memo, names="value")
    run.on_click(generate)
    for index, button in enumerate([result_button, html_button, zip_button]):
        button.on_click(lambda _, index=index: download(index))
    return w.VBox([title, material, *fields.values(), audience, rank_select, upload, run,
                   w.HBox([html_button, result_button, zip_button], layout=w.Layout(flex_flow="row wrap")), output])

studio = mount_studio()
display(studio)


In [ ]:
assert len(train) == 24 and [c["id"] for c in validate_cases] == ["BAG-07", "BAG-10", "TEE-11"]
assert all(item["origin"] == "replay" for item in reference)
assert measures(next(item for item in reference if item["key"] == "BAG-07")["rows"][0]["raw"],
                validate_cases[0])["line_count"] == 8
assert next(item for item in reference if item["key"] == "TEE-11")["rows"][1]["repetition"] == 32
assert all(p.is_file() for p in downloads)
if training:
    assert all(len(report["updates"]) == 4 for report in training.values())
    assert all(report["changed_tensors"] > 0 for report in training.values())
    assert sum(item["kind"] == "rank-live" for item in items) == 3
    assert any(item["kind"] == "temperature" for item in items)
else:
    assert all(item["origin"] == "replay" for item in items)
print("W13B_R2_RUNTIME_PASS", "live" if training else "replay")
